In [ ]:
import requests
import pandas as pd

base = "http://localhost:52771/api"
project_id = "TRY"

features = requests.get(f"{base}/projects/{project_id}/aligned-features").json()

rows = []

for f in features:
    fid = f["alignedFeatureId"]
    name = f.get("name", "unknown")
    try:
        formulas = requests.get(
            f"{base}/projects/{project_id}/aligned-features/{fid}/formulas"
        ).json()
        if not formulas:
            continue
        best = max(formulas, key=lambda x: x["siriusScore"])
        formula_id = best["formulaId"]
        fp = requests.get(
            f"{base}/projects/{project_id}/aligned-features/{fid}/formulas/{formula_id}/fingerprint"
        ).json()
        if isinstance(fp, dict):
            fp = [fp[k] for k in sorted(fp.keys(), key=lambda x: int(x))]
        threshold = 0.5
        fp = [1 if x > threshold else 0 for x in fp]
        row = [name] + fp
        rows.append(row)
    
    except Exception as e:
        print(f"ignore {fid}，reason: {e}")
        continue

fp_len = len(rows[0]) - 1
columns = ["name"] + [i for i in range(fp_len)]

df = pd.DataFrame(rows, columns=columns)
df.to_csv(r"ECRFS_fps.csv", index=False)

print(f"success: {len(df)} rows, fingerprint dimension: {fp_len}")